## Collaspe to one row per game, from the home team's perspective

In [2]:
# preparations
import pandas as pd
rest_travel = pd.read_csv("rest_travel.csv")

In [3]:
# we extract home and away games and then merge them together by game_id
home = rest_travel[
    rest_travel["LOCATION_TYPE"] == "Home"
][[
    "GAME_ID",
    "SEASON",
    "GAME_DATE",
    "TEAM_ABBREVIATION",
    "PLUS_MINUS",
    "REST_DAYS",
    "TRAVEL_DISTANCE_MILES",
    "NBA_CUP_GAME",
    "ALL_STAR_BREAK"
]].copy()


away = rest_travel[
    rest_travel["LOCATION_TYPE"] == "Away"
][[
    "GAME_ID",
    "SEASON",
    "GAME_DATE",
    "TEAM_ABBREVIATION",
    "REST_DAYS",
    "TRAVEL_DISTANCE_MILES"
]].copy()

## Get rid of neutral site games, final output: single_game

In [4]:
# single_game = home.merge(away, on = ["GAME_ID", "SEASON", "GAME_DATE"], suffixes = ('_home', '_away'), validate = "one_to_one")

## raised an error because neutral site games were also labeled as away games for both teams, so the location type and travel distance doesn't really make sense;
## here we get rid of them by modifying away and then continue with the rest 

duplicate_away = away[
    away.duplicated(
        ["GAME_ID", "SEASON", "GAME_DATE"],
        keep=False
    )
].sort_values(["GAME_ID", "SEASON", "GAME_DATE"])

duplicate_ids = duplicate_away["GAME_ID"].unique()

# get rid of duplicated game_ids in away

away = away[
    ~away["GAME_ID"].astype(int).isin(duplicate_ids)
].copy()

In [5]:
# merge home and away 

single_game = home.merge(away, on = ["GAME_ID", "SEASON", "GAME_DATE"], suffixes = ('_home', '_away'), validate = "one_to_one")

In [6]:
# check the number of rows; should be 3(seasons) * 82(games per team) * 30(teams) / 2 - 10(neutral site games) = 3680
single_game.shape

(3680, 12)

In [7]:
single_game.to_csv("single_game.csv", index=False)